In [33]:
__author__ = 'Cyrille Mvomo, https://github.com/cyrillemvomo'
__version__ = "1"
__license__ = "MIT"

**Import**

In [1]:
import sys
sys.path.append("/Users/cyrilleetude/Documents/GitHub/WhiteBoxDL/Preprocessing/1_Get_Data/MJFF_Cohort/1_Transform")
import numpy as np
import pandas as pd
import pickle
from scipy.signal import  detrend
import copy, os

**Read**

In [2]:
# Declare
    # Signal 
        # Straight 
STRAIGHT_EXTRACTED_SIGNAL_PATH = "/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/RawData_Sourced/STRAIGHT_ON/"
STRAIGHT_EXTRACTED_DATA_FILES_LIST = np.sort(np.array(os.listdir(STRAIGHT_EXTRACTED_SIGNAL_PATH))) # e.g PD04.csv as for events so no need to re-declare
IDs_STRAIGHT = [filename[:6] for filename in STRAIGHT_EXTRACTED_DATA_FILES_LIST if not filename.startswith(('.', '~'))] # Participant ID's (drop the .csv)
IDs_STRAIGHT = [filename[:-1] if filename.endswith(('.')) else filename for filename in IDs_STRAIGHT]
SIGNAL_STRAIGHT = []
        # DT 
DT_EXTRACTED_SIGNAL_PATH = "/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/RawData_Sourced/DT_ON/"
DT_EXTRACTED_DATA_FILES_LIST = np.sort(np.array(os.listdir(DT_EXTRACTED_SIGNAL_PATH)))
IDs_DT = [filename[:6] for filename in DT_EXTRACTED_DATA_FILES_LIST if not filename.startswith(('.', '~'))] # Participant ID's
IDs_DT = [filename[:-1] if filename.endswith(('.')) else filename for filename in IDs_DT]
SIGNAL_DT = []

# Read 
for filename_straight in STRAIGHT_EXTRACTED_DATA_FILES_LIST:# Get files names
        if not filename_straight.startswith(('.', '~')):
                SIGNAL_STRAIGHT.append(pd.read_csv(os.path.join(STRAIGHT_EXTRACTED_SIGNAL_PATH, filename_straight), usecols=[0, 1]))
for filename_DT in DT_EXTRACTED_DATA_FILES_LIST:# Get files names
        if not filename_DT.startswith(('.', '~')):
                SIGNAL_DT.append(pd.read_csv(os.path.join(DT_EXTRACTED_SIGNAL_PATH, filename_DT), usecols=[0, 1]))

# Rename columns properly
for signal in SIGNAL_STRAIGHT:
        signal.columns = ["TIME_S", "ACC_VERTICAL_MS2"]
for signal in SIGNAL_DT:
        signal.columns = ["TIME_S", "ACC_VERTICAL_MS2"]


**Split into 5 second bouts**

In [8]:
# function to handle nan

def detrend_1d_array_with_nan(array):
    nan_indices = np.where(array * 0 !=0)[0]
    inf_indices = np.where(np.isinf(array))[0]
    nan_indices = np.unique(np.concatenate((nan_indices, inf_indices)))
    Data_no_nans = np.copy(array)
    Data_no_nans[nan_indices] = 0
    Data_no_nans = detrend(Data_no_nans)  # Detrend the data
    #Restore NaNs to their original positions
    Data_no_nans[nan_indices] = np.nan
    return Data_no_nans

- DT

In [11]:
# Split in 5 second bouts
EXTRACTED_DATA_SPLITTED_DT = {}
sampling_rate=50
for participant_id in range(len(IDs_DT)):
    five_s_bouts_indexes = np.array_split(SIGNAL_DT[participant_id].index.values, len(SIGNAL_DT[participant_id])//(5*sampling_rate))
    participant_data = []
    for bout_id in range(len(five_s_bouts_indexes)):
        participant_data.append(detrend_1d_array_with_nan(SIGNAL_DT[participant_id].iloc[five_s_bouts_indexes[bout_id]].ACC_VERTICAL_MS2.values))
    
    EXTRACTED_DATA_SPLITTED_DT[IDs_DT[participant_id]] = participant_data

In [ ]:
# Save as bin file 
with open("/Users/cyrilleetude/Documents/GitHub/WhiteBoxDL/Preprocessing/1_Get_Data/MJFF_Cohort/1_Transform/temp/Extracted_Data_MJFF_ON_DT.bin", "wb") as f:
            pickle.dump(EXTRACTED_DATA_SPLITTED_DT, f)

- Straight

In [18]:
# Split in 5 second bouts
EXTRACTED_DATA_SPLITTED_STRAIGHT = {}
sampling_rate=50
for participant_id in range(len(IDs_STRAIGHT)):
    five_s_bouts_indexes = np.array_split(SIGNAL_STRAIGHT[participant_id].index.values, len(SIGNAL_STRAIGHT[participant_id])//(5*sampling_rate))
    participant_data = []
    for bout_id in range(len(five_s_bouts_indexes)):
        participant_data.append(detrend_1d_array_with_nan(SIGNAL_STRAIGHT[participant_id].iloc[five_s_bouts_indexes[bout_id]].ACC_VERTICAL_MS2.values))
    
    EXTRACTED_DATA_SPLITTED_STRAIGHT[IDs_STRAIGHT[participant_id]] = participant_data

In [19]:
# Save as bin file 
with open("/Users/cyrilleetude/Documents/GitHub/WhiteBoxDL/Preprocessing/1_Get_Data/MJFF_Cohort/1_Transform/temp/Extracted_Data_MJFF_ON_STRAIGHT.bin", "wb") as f:
            pickle.dump(EXTRACTED_DATA_SPLITTED_STRAIGHT, f)